# Paquetes

In [5]:
import sys
!{sys.executable} -m pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import numpy as np
import roboticstoolbox as rtb
import matplotlib.pyplot as plt 
from tqdm.auto import tqdm

# Declaracion de robot

In [14]:
#Nota: con G != 1 la simulacion se cuelga y no se porque
dp = rtb.DHRobot(
    [
        rtb.RevoluteDH(a=0.2,m=1.5,
            r=np.array([-0.1, 0, 0]),
            I=np.array([0,0,0,0,0,0,0,0,1e-3]),
            B=1, G=1),
        rtb.RevoluteDH(a=0.2,m=1,
            r=np.array([-0.1, 0, 0]),
            I=np.array([0,0,0,0,0,0,0,0,1e-4]),
            B=1, G=1)
    ],
    gravity = np.array([0, -9.81, 0]), # Ojo con el signo, la gravedad va hacia abajo con signo positivo
    name="dp")

print(dp)
print(dp.dynamics())

DHRobot: dp, 2 joints (RR), dynamics, standard DH parameters
┌─────┬────┬─────┬──────┐
│ θⱼ  │ dⱼ │ aⱼ  │  ⍺ⱼ  │
├─────┼────┼─────┼──────┤
│  q1 │  0 │ 0.2 │ 0.0° │
│  q2 │  0 │ 0.2 │ 0.0° │
└─────┴────┴─────┴──────┘

┌──┬──┐
└──┴──┘

┌───────┬──────┬──────────────┬─────────────────────────────┬────┬────┬────────┬────┐
│   j   │  m   │      r       │              I              │ Jm │ B  │   Tc   │ G  │
├───────┼──────┼──────────────┼─────────────────────────────┼────┼────┼────────┼────┤
│ link1 │  1.5 │ -0.1,  0,  0 │  0,  0,  0.001,  0,  0,  0  │  0 │  1 │  0,  0 │  1 │
│ link2 │  1   │ -0.1,  0,  0 │  0,  0,  0.0001,  0,  0,  0 │  0 │  1 │  0,  0 │  1 │
└───────┴──────┴──────────────┴─────────────────────────────┴────┴────┴────────┴────┘

None


# Simulacion

In [15]:
solver_kwargs = {
    'rtol': 1e-4     # Tolerancia relativa
    #'atol': 1e-8     # Tolerancia absoluta
    #'max_step': 0.1   # Tamaño máximo del paso de integración
}

## Funciones Utiles

In [16]:
def plot_dinamica(tg, dp):
    # Plot de varaibles articulares
    plt.figure()
    plt.plot(tg.t,tg.q * 180/np.pi)
    plt.xlabel("Tiempo [s]")
    plt.ylabel("q [grados]")
    plt.show()

    plt.figure()
    plt.plot(tg.t,tg.qd * 180/np.pi)
    plt.xlabel("Tiempo [s]")
    plt.ylabel("q [grados]")
    plt.show()

    

    #Plot trayectoria
    plt.figure(figsize=(5,5))
    trayectoria = dp.fkine(tg.q)
    plt.plot(trayectoria.t[:,0],trayectoria.t[:,1],'b-',linewidth=1,label='real')
    plt.legend(loc='upper right')
    plt.xlabel('X [m]')
    plt.ylabel('Y [m]')
    plt.title('Trayectoria realizada')
    plt.axis([-0.4, 0.4, -0.4, 0.4])	
    plt.tight_layout()



## Condicion inicial con rozamiento

In [ ]:
q_ini = np.array([0,0])
qp_ini = np.array([0,0])

tg = dp.nofriction(coulomb=True, viscous=False).fdyn(10, q_ini, qd0 = qp_ini, solver_args= solver_kwargs, dt=0.005)

plot_dinamica(tg, dp)

## Poco rozamiento

In [ ]:
#Modifico la viscosidad del modelo
dp.links[0].B = 0.1
dp.links[1].B = 0.1

q_ini = np.array([0,0])
qp_ini = np.array([0,0])

tg = dp.nofriction(coulomb=True, viscous=False).fdyn(10, q_ini, qd0 = qp_ini, solver_args= solver_kwargs, dt=0.005)

plot_dinamica(tg, dp)

## Condicion inicial sin rozamiento (caotico)

In [ ]:
#Modifico la viscosidad del modelo
dp.links[0].B = 0
dp.links[1].B = 0

q_ini = np.array([0,0])
qp_ini = np.array([0,0])

tg = dp.nofriction(coulomb=True, viscous=False).fdyn(10, q_ini, qd0 = qp_ini, solver_args= solver_kwargs, dt=0.005)

plot_dinamica(tg, dp)

## Equilibrio Inestable

In [ ]:
#Modifico la viscosidad del modelo
dp.links[0].B = 1
dp.links[1].B = 1

q_ini = np.array([np.pi/2,0])
qp_ini = np.array([0,0])

tg = dp.nofriction(coulomb=True, viscous=False).fdyn(30, q_ini, qd0 = qp_ini, solver_args= solver_kwargs, dt=0.005)

plot_dinamica(tg, dp)

# Lazo cerrado

## Modelo de la planta

La dinámica del robot en la forma de Lagrange es:

$$\boldsymbol{\tau} = \mathbf{M}(\mathbf{q})\,\ddot{\mathbf{q}} + \mathbf{H}(\mathbf{q},\dot{\mathbf{q}}) + \mathbf{G}(\mathbf{q})$$

despejando $\ddot{\mathbf{q}}$:

$$\ddot{\mathbf{q}} = \mathbf{M}^{-1}(\mathbf{q})\,[\boldsymbol{\tau} - \mathbf{H}(\mathbf{q},\dot{\mathbf{q}}) - \mathbf{G}(\mathbf{q})]$$

con:

$$\mathbf{M}(\mathbf{q}) = 
\begin{bmatrix}
I_{01zz} + 2a_1x_{g1}m_1 + a_1^2(m_1+m_2) + 2m_{12} - m_{22} & m_{12} \\
m_{12} & I_{02zz} + 2a_2x_{g2}m_2 + a_2^2m_2
\end{bmatrix}$$

$$\mathbf{H}(\mathbf{q},\dot{\mathbf{q}}) = -a_1[(a_2+x_{g2})s_2 + y_{g2}c_2]\,m_2
\begin{bmatrix} 2\dot{q}_1\dot{q}_2 + \dot{q}_2^2 \\ -\dot{q}_1^2 \end{bmatrix}$$

$$\mathbf{G}(\mathbf{q}) = 
\begin{bmatrix}
m_1 g[(x_{g1}+a_1)c_1 - y_{g1}s_1] + m_2 g a_1 c_1 + g_2 \\
m_2 g[(x_{g2}+a_2)c_{12} - y_{g2}s_{12}]
\end{bmatrix}$$

El sistema es **MIMO, no lineal y acoplado**: $\boldsymbol{\tau}$ afecta a ambos ejes simultáneamente a través de $\mathbf{M}(\mathbf{q})$.

## Control por par calculado (linealización exacta)

### Separación MIMO → n canales SISO

El sistema es MIMO acoplado, por lo que no se pueden aplicar directamente las técnicas de diseño SISO. La teoría (diap. 64–67) propone una **linealización exacta por realimentación de estado**, conocida como **control por torque computado**.

La dinámica se escribe en la forma $\dot{\mathbf{X}} = \mathbf{f}(\mathbf{X}) + g(\mathbf{X})\,\boldsymbol{\tau}$ con $g = \mathbf{M}^{-1}$. Se elige el torque como:

$$\boldsymbol{\tau} = g^{-1}[\mathbf{v} - \mathbf{f}] = \mathbf{M}(\mathbf{q})\,\mathbf{v} + \mathbf{h}(\mathbf{q},\dot{\mathbf{q}})$$

donde $\mathbf{h} = \mathbf{H} + \mathbf{G}$. Sustituyendo:

$$\mathbf{M}\ddot{\mathbf{q}} = \mathbf{M}\,\mathbf{v} + \cancel{\mathbf{h}} - \cancel{\mathbf{h}} \implies \boxed{\ddot{\mathbf{q}} = \mathbf{v}}$$

La planta queda **linealizada y desacoplada** en $n$ integradores dobles SISO independientes. Cada eje puede diseñarse por separado.

### Diseño del controlador PD

Sobre cada integrador doble se aplica una ley PD con feedforward de aceleración (diap. 68–69):

$$v_i = \ddot{q}_{ref,i} + K_{d,i}\,\dot{e}_i + K_{p,i}\,e_i \qquad (e_i = q_{ref,i} - q_i)$$

La dinámica del error en lazo cerrado es $\ddot{e}_i + K_d\,\dot{e}_i + K_p\,e_i = 0$, cuyo polinomio característico compara con $s^2 + 2\zeta\omega_n s + \omega_n^2 = 0$.

Las fórmulas de sintonización de la teoría (diap. 32), con $J_{ef}=1$, $B_{ef}=0$, $N=1$, $K_m=1$ tras la linealización exacta, se reducen a:

$$\boxed{K_p = \omega_n^2 \qquad K_d = 2\,\omega_n} \qquad (\zeta = 1,\;\text{críticamente amortiguado})$$

**Restricción de muestreo** (diap. 30): $\omega_n < \dfrac{1}{20}\dfrac{2\pi}{T_s}$. Con $T_s=0.005\,\text{s}$: $\omega_n < 62.8\,\text{rad/s}$.

La ley de control completa es (diap. 69):

$$\boldsymbol{\tau} = \mathbf{M}(\mathbf{q})\bigl[\ddot{\mathbf{q}}_{ref} + K_p(\mathbf{q}_{ref}-\mathbf{q}) + K_d(\dot{\mathbf{q}}_{ref}-\dot{\mathbf{q}})\bigr] + \mathbf{h}(\mathbf{q},\dot{\mathbf{q}})$$

In [ ]:
wn   = 5.0   # frecuencia natural [rad/s] — elegido dentro del límite de muestreo
zeta = 1.0   # críticamente amortiguado (diap. 21)

Kp = wn**2        # = 25
Kd = 2 * wn       # = 10  (con zeta=1: Kd = 2*zeta*wn = 2*wn)

print(f"Kp = {Kp},  Kd = {Kd}")
print(f"ts ≈ {4/wn:.2f} s  (4/ωn, sin sobrepico)")


def control_par_calculado(robot, t, q, qd, q_ref, qd_ref=None, qdd_ref=None):
    if qd_ref  is None: qd_ref  = np.zeros_like(q)
    if qdd_ref is None: qdd_ref = np.zeros_like(q)

    e  = q_ref  - q
    de = qd_ref - qd
    v  = qdd_ref + Kd * de + Kp * e      # entrada virtual por eje

    M   = robot.inertia(q)
    H   = robot.coriolis(q, qd) @ qd
    G   = robot.gravload(q)
    return M @ v + H + G                  # tau = M*v + H + G


# Ensayo: escalón a [45°, 45°]
q_ref  = np.array([np.pi/4, np.pi/4])
dp.links[0].B = dp.links[1].B = 1

tg = dp.nofriction(coulomb=True, viscous=False).fdyn(
    5, np.zeros(2),
    Q=lambda robot, t, q, qd: control_par_calculado(robot, t, q, qd, q_ref),
    qd0=np.zeros(2), solver_args=solver_kwargs, dt=0.005
)

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(tg.t, np.degrees(tg.q[:, i]), label=f'q{i+1}')
    ax.axhline(np.degrees(q_ref[i]), color='r', ls='--', label='ref')
    ax.set_ylabel(f'q{i+1} [°]'); ax.legend(); ax.grid(True)
axes[-1].set_xlabel('Tiempo [s]')
fig.suptitle(f'Par calculado — ωn={wn} rad/s, ζ={zeta}')
plt.tight_layout(); plt.show()

## Control PD con compensación de gravedad (PD + FF)

La ley de control agrega un feedforward de aceleración (escalado por la inercia nominal) y uno gravitacional:

$$\boldsymbol{\tau} = \mathbf{M}_{nom}\,\ddot{\mathbf{q}}_{ref} + K_p\,\mathbf{e} + K_d\,\dot{\mathbf{e}} + \mathbf{G}(\mathbf{q})$$

A diferencia del par calculado, **no se cancela $\mathbf{M}(\mathbf{q})$ ni $\mathbf{H}$** — el sistema sigue siendo no lineal. Se usa $\mathbf{M}_{nom}$ constante (evaluada en el punto de operación).

### Sintonización con las fórmulas de la teoría (diap. 32)

$$\boxed{K_{p,i} = \frac{\omega_n^2\,J_{ef,i}}{NK_m}} \qquad \boxed{K_{d,i} = \frac{2\sqrt{NK_mK_{p,i}J_{ef,i}} - B_{ef,i}}{NK_m}}$$

Para nuestro modelo sin motor ($N=1$, $K_m=1$) y con fricción viscosa $B_{ef}=B$:

$$K_{p,i} = \omega_n^2\,m_{ii}(\mathbf{q}_{nom}) \qquad K_{d,i} = 2\,\omega_n\,m_{ii}(\mathbf{q}_{nom}) - B_i$$

Los parámetros del robot evaluados en el punto de operación $\mathbf{q}_{nom} = [\pi/4,\,\pi/4]$:

| Eje | $m_{ii}$ [kg·m²] | $B_i$ [N·m·s/rad] |
|-----|-----------------|-------------------|
|  1  |  ≈ 0.094        |  1                |
|  2  |  0.0101         |  1                |

> **Nota:** si $K_{d,i} < 0$ es porque la fricción viscosa ya amortigua más de lo necesario — en ese caso se fija $K_{d,i} = 0$.

Restricción de muestreo: $\omega_n < \frac{1}{20}\frac{2\pi}{T_s} \approx 62.8\,\text{rad/s}$. Con $\omega_n=5$ estamos muy dentro del límite.

In [ ]:
## Sintonización PD+FF — fórmulas de la teoría (diap. 32)
wn   = 5.0   # frecuencia natural deseada [rad/s]
N    = 1.0   # relación de reducción (G=1 en el modelo)
Km   = 1.0   # constante de par (control directo de torque)
B    = 1.0   # fricción viscosa por eje [N·m·s/rad]
Ts   = 0.005 # período de muestreo [s]

# Verificación restricción de muestreo (diap. 30)
wn_max = (1/20) * (2*np.pi / Ts)
print(f"Límite ωn por muestreo: {wn_max:.1f} rad/s  →  ωn={wn} rad/s  {'OK' if wn < wn_max else 'EXCEDIDO'}")

# Inercia efectiva nominal por eje en q_nom = [pi/4, pi/4]
q_nom = np.array([np.pi/4, np.pi/4])
M_q_nom = dp.inertia(q_nom)
Jef = np.array([M_q_nom[0, 0], M_q_nom[1, 1]])   # diagonal de M en q_nom
print(f"Jef = {Jef}")

# Fórmulas teoría (diap. 32): Kp = wn²*Jef/(N*Km),  Kd = (2*sqrt(N*Km*Kp*Jef) - Bef) / (N*Km)
Kp_vec = (wn**2 * Jef) / (N * Km)
Kd_vec = (2 * np.sqrt(N * Km * Kp_vec * Jef) - B) / (N * Km)
Kd_vec = np.maximum(0, Kd_vec)   # la fricción existente puede suplir el amortiguamiento

print(f"Kp = diag{np.round(Kp_vec, 4)}")
print(f"Kd = diag{np.round(Kd_vec, 4)}")

# Matrices de ganancias (diagonales — sin acoplamiento)
Kp_pd = np.diag(Kp_vec)
Kd_pd = np.diag(Kd_vec)
M_nom = np.diag(Jef)   # feedforward de aceleración con inercia nominal


def control_pd_gravedad(robot, t, q, qd, q_ref, qd_ref=None, qdd_ref=None):
    """
    PD con FF gravitacional y FF de aceleración nominal.
    tau = M_nom @ qdd_ref + Kp*(q_ref - q) + Kd*(qd_ref - qd) + G(q)
    Kp, Kd sintonizados con fórmulas de diap. 32.
    """
    if qd_ref  is None: qd_ref  = np.zeros_like(q)
    if qdd_ref is None: qdd_ref = np.zeros_like(q)

    e  = q_ref - q
    de = qd_ref - qd

    G = robot.gravload(q)

    tau = M_nom @ qdd_ref + Kp_pd @ e + Kd_pd @ de + G
    return tau


# --- Ensayo: referencia escalón a [pi/4, pi/4] (qdd_ref = 0) ---
q_ref_pd = np.array([np.pi/4, np.pi/4])
control_fn_pd = lambda robot, t, q, qd: control_pd_gravedad(robot, t, q, qd, q_ref_pd)

dp.links[0].B = 1
dp.links[1].B = 1

tg_pd = dp.nofriction(coulomb=True, viscous=False).fdyn(
    5, np.zeros(2), Q=control_fn_pd, qd0=np.zeros(2), solver_args=solver_kwargs, dt=0.005
)

fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(tg_pd.t, np.degrees(tg_pd.q[:, i]), label=f'q{i+1} real')
    ax.axhline(np.degrees(q_ref_pd[i]), color='r', linestyle='--', label=f'q{i+1} ref')
    ax.set_ylabel(f'q{i+1} [°]')
    ax.legend(); ax.grid(True)
axes[-1].set_xlabel('Tiempo [s]')
fig.suptitle(f'PD + FF gravitacional — $\\omega_n$={wn} rad/s  (diap. 32)')
plt.tight_layout(); plt.show()